In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def setup_custom_vllm():
    print("Searching for locally attached vLLM wheels (from AIMO 3 notebook)...")
    wheel_paths = list(Path('/kaggle/input').rglob('wheels.tar.gz'))
    
    if not wheel_paths:
        print("❌ Could not find wheels.tar.gz. Falling back to standard older vLLM version without V1 engine bug...")
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-U', '-q', 'vllm==0.6.3', 'pyngrok', 'requests'])
        return
        
    archive_path = str(wheel_paths[0])
    temp_dir = '/kaggle/tmp/setup'
    print(f"Found custom wheels at {archive_path}. Extracting...")
    
    if not os.path.exists(temp_dir):
        os.makedirs(temp_dir, exist_ok=True
        subprocess.run(['tar', '-xzf', archive_path, '-C', temp_dir], check=True)
        
    print("Installing custom vLLM build...")
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', 
        f'{temp_dir}/wheels', 'vllm'
    ], check=True)
    
    # Also install ngrok and requests since the wheel archive might not have them
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok', 'requests'], check=True)
    print("✅ Custom vLLM environment installed successfully!")

setup_custom_vllm()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 MB 4.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 117.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 100.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 2.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 42.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━

In [2]:

# # ================================================
# # Kaggle GPU + vLLM + ngrok Deploy Script
# # GUI-Owl-1.5-8B-Instruct (mPLUG, Qwen3-VL-based GUI agent)
# # ================================================

# import os
# import subprocess
# import time
# import sys
# from pyngrok import ngrok

# # ================== CONFIG ==================
# MODEL_ID = "mPLUG/GUI-Owl-1.5-8B-Instruct"  # GUI-Owl 1.5 8B — Qwen3-VL-based multi-platform GUI agent
# NGROK_TOKEN = "32bCqPHMLLYElvEw0OT75n4N7vs_4ev6MRxztr1sGfv8PDeJJ"  # Replace with your token!
# PORT = 8000

# # Multimodal limit as PROPER JSON string
# MM_LIMIT = '{"image": 5}'  # Good default for VL models; increase if needed

# EXTRA_FLAGS = [
#     "--gpu-memory-utilization", "0.90",
#     "--max-model-len", "16384",
#     "--trust-remote-code",
#     "--host", "0.0.0.0",
#     "--port", str(PORT),
#     "--served-model-name", "gui-owl-1.5-8b-instruct",
#     "--tensor-parallel-size", "1",
#     "--limit-mm-per-prompt", MM_LIMIT,
# ]


In [3]:

# print(f"Starting vLLM for GUI-Owl-1.5-8B-Instruct: {MODEL_ID}")

# import os
# os.environ["VLLM_LOGGING_LEVEL"] = "DEBUG"
# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

# # ================== START SERVER ==================
# import subprocess, time, requests

# cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
#        "--model", MODEL_ID] + EXTRA_FLAGS

# log_file = open("vllm.log", "w")
# proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file, text=True)

# print("vLLM PID:", proc.pid, "→ tail -f vllm.log to watch")

# print("Waiting up to 15 min for /health...")
# for i in range(180):
#     time.sleep(5)
#     try:
#         r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
#         if r.status_code == 200:
#             print("✅ Server is READY!")
#             break
#     except:
#         if i % 6 == 0:
#             print(f"({i*5}s) Still waiting... last logs:")
#             !tail -n 8 vllm.log
#             print("-"*50)

# else:
#     print("❌ Timeout. Check logs:")
#     !tail -n 50 vllm.log
#     proc.terminate()
#     raise SystemExit("Failed")

# # ================== NGROK ==================
# from pyngrok import ngrok
# ngrok.set_auth_token(NGROK_TOKEN)
# tunnel = ngrok.connect(PORT, "http")
# print("\n" + "="*60)
# print("🎉 LIVE ENDPOINT:", tunnel.public_url + "/v1")
# print("Model name: gui-owl-1.5-8b-instruct")
# print("Use in code: OpenAI(base_url=..., api_key='anything')")
# print("="*60)

# # Keep alive
# try:
#     while True:
#         time.sleep(30)
# except KeyboardInterrupt:
#     print("Shutting down...")
#     ngrok.disconnect(tunnel.public_url)
#     proc.terminate()


In [ ]:
# ======================================================================
# Kaggle + H100 Ready vLLM + ngrok Deploy Script
# Standard Text Inference for Qwen 27B
# ======================================================================

import os
import subprocess
import time
import sys
from pyngrok import ngrok
import requests
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

# ================== ENVIRONMENT ALIGNMENT (from AIMO3) ==================
os.environ['TRANSFORMERS_NO_TF'] = '1'
os.environ['TRANSFORMERS_NO_FLAX'] = '1'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TRITON_PTXAS_PATH'] = '/usr/local/cuda/bin/ptxas'

# ================== CONFIGURATION ==================
# Auto-discover the local Kaggle model path instead of downloading from HuggingFace
print("Searching for local model in /kaggle/input...")
local_paths = list(Path('/kaggle/input').rglob('config.json'))
if local_paths:
    MODEL_ID = str(local_paths[0].parent)
    print(f"Found local model at: {MODEL_ID}")
else:
    # Fallback if rglob fails but we know the path from your other notebook
    MODEL_ID = '/kaggle/input/models/qwen-lm/qwen-3-5/transformers/qwen3.5-27b/1'
    print(f"Could not search, defaulting to known local path: {MODEL_ID}")

# Replace with your actual Ngrok Auth Token from dashboard.ngrok.com
NGROK_TOKEN = "32bCqPHMLLYElvEw0OT75n4N7vs_4ev6MRxztr1sGfv8PDeJJ"  
PORT = 8000

# H100 Optimized Flags for Text-Only Inference (Matched to AIMO3)
EXTRA_FLAGS = [
    "--gpu-memory-utilization", "0.96",  
    "--max-model-len", "65536",          
    "--dtype", "auto",               
    "--kv-cache-dtype", "fp8_e4m3",      
    "--max-num-seqs", "256",
    "--trust-remote-code",
    "--host", "0.0.0.0",
    "--port", str(PORT),
    "--served-model-name", "qwen3.5-27b",
    "--tensor-parallel-size", "1",
    "--disable-log-stats",
    "--enable-prefix-caching",
    "--async-scheduling",
    "--language-model-only"              # CRITICAL: prevents vision encoder from OOM crashing during boot
]

def preload_model_weights():
    print(f'Loading model weights from {MODEL_ID} into OS Page Cache...')
    start_time = time.time()
    files_to_load = []
    total_size = 0
    for root, _, files in os.walk(MODEL_ID):
        for file_name in files:
            file_path = os.path.join(root, file_name)
            if os.path.isfile(file_path):
                files_to_load.append(file_path)
                total_size += os.path.getsize(file_path)
                
    def _read_file(path: str) -> None:
        with open(path, 'rb') as file_object:
            while file_object.read(1024 * 1024 * 1024):
                pass
                
    with ThreadPoolExecutor(max_workers=16) as executor:
        list(executor.map(_read_file, files_to_load))
    elapsed = time.time() - start_time
    print(f'Processed {len(files_to_load)} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

preload_model_weights()
print(f"Starting vLLM API Server for: {MODEL_ID}")

# ================== START SERVER ==================
os.environ["VLLM_LOGGING_LEVEL"] = "INFO"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# CRITICAL: Disable the unstable V1 engine that gets stuck on Kaggle H100
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_ENABLE_V1"] = "0"

cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
       "--model", MODEL_ID,
       "--disable-v1-engine"] + EXTRA_FLAGS

log_file = open("vllm.log", "w")
proc = subprocess.Popen(cmd, stdout=log_file, stderr=log_file, text=True)

print(f"vLLM PID: {proc.pid} → Server is booting. This will take a few minutes to load weights.")
print("Waiting for /health endpoint...")

# Health Check Polling
for i in range(180): # Wait up to 15 minutes
    time.sleep(5)
    try:
        r = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=3)
        if r.status_code == 200:
            print("✅ Server is READY!")
            break
    except requests.exceptions.RequestException:
        if i % 6 == 0:
            print(f"({i*5}s) Still loading weights... checking logs:")
            os.system("tail -n 5 vllm.log")
            print("-" * 50)
else:
    print("❌ Timeout reached. The server failed to start. Check logs:")
    os.system("tail -n 50 vllm.log")
    proc.terminate()
    raise SystemExit("Deployment Failed")

# ================== NGROK TUNNEL ==================
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(PORT, "http")

print("\n" + "="*70)
print(f"🎉 LIVE API ENDPOINT: {tunnel.public_url}/v1")
print(f"🤖 Model Name parameter: qwen3.5-27b")
print("="*70)

# Keep the Kaggle cell running so the server doesn't shut down
try:
    print("Server is running. Press Stop in Kaggle to terminate.")
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    print("\nShutting down server and tunnel...")
    ngrok.disconnect(tunnel.public_url)
    proc.terminate()

Starting vLLM API Server for: Qwen/Qwen3.5-27B
vLLM PID: 342 → Server is booting. This will take a few minutes to load weights.
Waiting for /health endpoint...
(0s) Still loading weights... checking logs:
W0000 00:00:1772659452.589075     342 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772659452.589076     342 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'
--------------------------------------------------
(30s) Still loading weights... checking logs:
(APIServer pid=342) pydantic_core._pydantic_core.ValidationError: 1 validation error for ModelConfig
(APIServer pid=342)   Value err